In [2]:
!pip install pandas h3 pyarrow tqdm

In [4]:
import pandas as pd
import numpy as np
import h3
from tqdm import tqdm
from collections import defaultdict
from datetime import datetime


In [7]:
df = pd.read_csv('../chicago crimes.csv')
df.head()

/var/folders/dh/t016p6ms66zc3_d2crms5hlw0000gn/T/ipykernel_28624/2260344088.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../chicago crimes.csv')


,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,...,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,13311263,JG503434,07/29/2022 03:39:00 AM,023XX S TROY ST,1582,OFFENSE INVOLVING CHILDREN,CHILD PORNOGRAPHY,RESIDENCE,True,False,...,25.0,30.0,17,NaN,NaN,2022,04/18/2024 03:40:59 PM,NaN,NaN,NaN
1,13053066,JG103252,01/03/2023 04:44:00 PM,039XX W WASHINGTON BLVD,2017,NARCOTICS,MANUFACTURE / DELIVER - CRACK,SIDEWALK,True,False,...,28.0,26.0,18,NaN,NaN,2023,01/20/2024 03:41:12 PM,NaN,NaN,NaN
2,12131221,JD327000,08/10/2020 09:45:00 AM,015XX N DAMEN AVE,0326,ROBBERY,AGGRAVATED VEHICULAR HIJACKING,STREET,True,False,...,1.0,24.0,03,1162795.0,1909900.0,2020,05/17/2025 03:40:52 PM,41.908418,-87.677407,"(41.908417822, -87.67740693)"
3,11227634,JB147599,08/26/2017 10:00:00 AM,001XX W RANDOLPH ST,0281,CRIM SEXUAL ASSAULT,NON-AGGRAVATED,HOTEL/MOTEL,False,False,...,42.0,32.0,02,NaN,NaN,2017,02/11/2018 03:57:41 PM,NaN,NaN,NaN
4,13203321,JG415333,09/06/2023 05:00:00 PM,002XX N Wells st,1320,CRIMINAL DAMAGE,TO VEHICLE,PARKING LOT / GARAGE (NON RESIDENTIAL),False,False,...,42.0,32.0,14,1174694.0,1901831.0,2023,11/04/2023 03:40:18 PM,41.886018,-87.633938,"(41.886018055, -87.633937881)"


In [8]:
df.columns = (
    df.columns.str.strip()
      .str.lower()
      .str.replace(r"[^a-z0-9]+", "_", regex=True)
      .str.strip("_")
)

In [9]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["updated_on"] = pd.to_datetime(df["updated_on"], errors="coerce")

df.head()

/var/folders/dh/t016p6ms66zc3_d2crms5hlw0000gn/T/ipykernel_28624/3367665329.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["updated_on"] = pd.to_datetime(df["updated_on"], errors="coerce")


,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,...,ward,community_area,fbi_code,x_coordinate,y_coordinate,year,updated_on,latitude,longitude,location
0,13311263,JG503434,2022-07-29 03:39:00,023XX S TROY ST,1582,OFFENSE INVOLVING CHILDREN,CHILD PORNOGRAPHY,RESIDENCE,True,False,...,25.0,30.0,17,NaN,NaN,2022,2024-04-18 15:40:59,NaN,NaN,NaN
1,13053066,JG103252,2023-01-03 16:44:00,039XX W WASHINGTON BLVD,2017,NARCOTICS,MANUFACTURE / DELIVER - CRACK,SIDEWALK,True,False,...,28.0,26.0,18,NaN,NaN,2023,2024-01-20 15:41:12,NaN,NaN,NaN
2,12131221,JD327000,2020-08-10 09:45:00,015XX N DAMEN AVE,0326,ROBBERY,AGGRAVATED VEHICULAR HIJACKING,STREET,True,False,...,1.0,24.0,03,1162795.0,1909900.0,2020,2025-05-17 15:40:52,41.908418,-87.677407,"(41.908417822, -87.67740693)"
3,11227634,JB147599,2017-08-26 10:00:00,001XX W RANDOLPH ST,0281,CRIM SEXUAL ASSAULT,NON-AGGRAVATED,HOTEL/MOTEL,False,False,...,42.0,32.0,02,NaN,NaN,2017,2018-02-11 15:57:41,NaN,NaN,NaN
4,13203321,JG415333,2023-09-06 17:00:00,002XX N Wells st,1320,CRIMINAL DAMAGE,TO VEHICLE,PARKING LOT / GARAGE (NON RESIDENTIAL),False,False,...,42.0,32.0,14,1174694.0,1901831.0,2023,2023-11-04 15:40:18,41.886018,-87.633938,"(41.886018055, -87.633937881)"


In [10]:
def to_bool(s):
    # handles True/False, "True"/"False", 1/0, "Y"/"N"
    if s.dtype == bool:
        return s
    return (
        s.astype(str)
         .str.strip()
         .str.lower()
         .map({"true": True, "false": False, "1": True, "0": False, "y": True, "n": False})
    )

for col in ["arrest", "domestic"]:
    if col in df.columns:
        df[col] = to_bool(df[col])

# numeric columns (safe coercion)
num_cols = ["ward", "community_area", "x_coordinate", "y_coordinate", "year", "latitude", "longitude"]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [11]:
before = len(df)

df_clean = df.dropna(subset=["date"])  # must have time
df_clean = df_clean.dropna(subset=["latitude", "longitude"])  # must have coords

# optional: keep only chicago-ish bounds to remove bad points
df_clean = df_clean[
    (df_clean["latitude"].between(41.60, 42.10)) &
    (df_clean["longitude"].between(-88.10, -87.40))
]

after = len(df_clean)
print(f"Rows before: {before:,} | after clean: {after:,} | dropped: {before-after:,} ({(before-after)/before:.1%})")
df_clean.head()

Rows before: 8,500,901 | after clean: 8,406,081 | dropped: 94,820 (1.1%)


,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,...,ward,community_area,fbi_code,x_coordinate,y_coordinate,year,updated_on,latitude,longitude,location
2,12131221,JD327000,2020-08-10 09:45:00,015XX N DAMEN AVE,0326,ROBBERY,AGGRAVATED VEHICULAR HIJACKING,STREET,True,False,...,1.0,24.0,03,1162795.0,1909900.0,2020,2025-05-17 15:40:52,41.908418,-87.677407,"(41.908417822, -87.67740693)"
4,13203321,JG415333,2023-09-06 17:00:00,002XX N Wells st,1320,CRIMINAL DAMAGE,TO VEHICLE,PARKING LOT / GARAGE (NON RESIDENTIAL),False,False,...,42.0,32.0,14,1174694.0,1901831.0,2023,2023-11-04 15:40:18,41.886018,-87.633938,"(41.886018055, -87.633937881)"
5,13204489,JG416325,2023-09-06 11:00:00,0000X E 8TH ST,0810,THEFT,OVER $500,PARKING LOT / GARAGE (NON RESIDENTIAL),False,False,...,4.0,32.0,06,1176857.0,1896680.0,2023,2023-11-04 15:40:18,41.871835,-87.626151,"(41.871834768, -87.62615082)"
6,11695116,JC272771,2019-05-21 08:20:00,018XX S CALIFORNIA AVE,0620,BURGLARY,UNLAWFUL ENTRY,RESIDENCE,False,False,...,25.0,29.0,05,1157982.0,1890961.0,2019,2024-01-19 15:40:50,41.856547,-87.695605,"(41.856547057, -87.695604526)"
7,12419690,JE295655,2021-07-07 10:30:00,132XX S GREENWOOD AVE,1544,SEX OFFENSE,SEXUAL EXPLOITATION OF A CHILD,RESIDENCE,False,False,...,10.0,54.0,17,1186051.0,1817781.0,2021,2024-01-19 15:40:50,41.655116,-87.594883,"(41.65511579, -87.594883198)"


In [12]:
if "id" in df_clean.columns:
    df_clean = df_clean.drop_duplicates(subset=["id"])
elif "case_number" in df_clean.columns:
    df_clean = df_clean.drop_duplicates(subset=["case_number", "date"])

print("After dedupe:", len(df_clean))

After dedupe: 8406081


In [13]:
df_clean["hour"] = df_clean["date"].dt.hour
df_clean["dow"] = df_clean["date"].dt.dayofweek  # Mon=0
df_clean["hour_of_week"] = df_clean["dow"] * 24 + df_clean["hour"]

df_clean["is_weekend"] = df_clean["dow"].isin([5, 6])
df_clean["is_night"] = (df_clean["hour"] >= 20) | (df_clean["hour"] <= 5)

df_clean[["date", "hour", "dow", "hour_of_week", "is_weekend", "is_night"]].head()

,date,hour,dow,hour_of_week,is_weekend,is_night
2,2020-08-10 09:45:00,9,0,9,False,False
4,2023-09-06 17:00:00,17,2,65,False,False
5,2023-09-06 11:00:00,11,2,59,False,False
6,2019-05-21 08:20:00,8,1,32,False,False
7,2021-07-07 10:30:00,10,2,58,False,False


In [14]:
print("Date range:", df_clean["date"].min(), "→", df_clean["date"].max())
print("Unique primary types:", df_clean["primary_type"].nunique() if "primary_type" in df_clean.columns else "N/A")

# missingness check
df_clean.isna().mean().sort_values(ascending=False).head(10)

Date range: 2001-01-01 00:00:00 → 2026-02-16 00:00:00
Unique primary types: 34


ward                    0.072035
community_area          0.071906
location_description    0.001215
district                0.000006
x_coordinate            0.000000
is_weekend              0.000000
hour_of_week            0.000000
dow                     0.000000
hour                    0.000000
location                0.000000
dtype: float64

In [15]:
primary_types = sorted(df_clean["primary_type"].dropna().unique())

for pt in primary_types:
    print(pt)

print("\nTotal primary types:", len(primary_types))
df_clean["primary_type"].value_counts().sort_values(ascending=False)

ARSON
ASSAULT
BATTERY
BURGLARY
CONCEALED CARRY LICENSE VIOLATION
CRIM SEXUAL ASSAULT
CRIMINAL DAMAGE
CRIMINAL SEXUAL ASSAULT
CRIMINAL TRESPASS
DECEPTIVE PRACTICE
DOMESTIC VIOLENCE
GAMBLING
HOMICIDE
HUMAN TRAFFICKING
INTERFERENCE WITH PUBLIC OFFICER
INTIMIDATION
KIDNAPPING
LIQUOR LAW VIOLATION
MOTOR VEHICLE THEFT
NARCOTICS
NON-CRIMINAL
OBSCENITY
OFFENSE INVOLVING CHILDREN
OTHER NARCOTIC VIOLATION
OTHER OFFENSE
PROSTITUTION
PUBLIC INDECENCY
PUBLIC PEACE VIOLATION
RITUALISM
ROBBERY
SEX OFFENSE
STALKING
THEFT
WEAPONS VIOLATION

Total primary types: 34


primary_type
THEFT                                1784232
BATTERY                              1541831
CRIMINAL DAMAGE                       961721
NARCOTICS                             753109
ASSAULT                               568575
OTHER OFFENSE                         525992
BURGLARY                              447119
MOTOR VEHICLE THEFT                   433329
DECEPTIVE PRACTICE                    371147
ROBBERY                               314622
CRIMINAL TRESPASS                     227435
WEAPONS VIOLATION                     125780
PROSTITUTION                           69793
OFFENSE INVOLVING CHILDREN             56505
PUBLIC PEACE VIOLATION                 54850
SEX OFFENSE                            32529
CRIM SEXUAL ASSAULT                    25786
INTERFERENCE WITH PUBLIC OFFICER       20457
LIQUOR LAW VIOLATION                   15313
GAMBLING                               14568
ARSON                                  14430
HOMICIDE                               140

In [16]:
locations = sorted(df_clean["location_description"].dropna().unique())

for loc in locations:
    print(loc)

print("\nTotal unique location descriptions:", len(locations))
df_clean["location_description"].value_counts().sort_values(ascending=False)

ABANDONED BUILDING
AIRCRAFT
AIRPORT BUILDING NON-TERMINAL - NON-SECURE AREA
AIRPORT BUILDING NON-TERMINAL - SECURE AREA
AIRPORT EXTERIOR - NON-SECURE AREA
AIRPORT EXTERIOR - SECURE AREA
AIRPORT PARKING LOT
AIRPORT TERMINAL LOWER LEVEL - NON-SECURE AREA
AIRPORT TERMINAL LOWER LEVEL - SECURE AREA
AIRPORT TERMINAL MEZZANINE - NON-SECURE AREA
AIRPORT TERMINAL UPPER LEVEL - NON-SECURE AREA
AIRPORT TERMINAL UPPER LEVEL - SECURE AREA
AIRPORT TRANSPORTATION SYSTEM (ATS)
AIRPORT VENDING ESTABLISHMENT
AIRPORT/AIRCRAFT
ALLEY
ANIMAL HOSPITAL
APARTMENT
APPLIANCE STORE
ATHLETIC CLUB
ATM (AUTOMATIC TELLER MACHINE)
AUTO
AUTO / BOAT / RV DEALERSHIP
BANK
BANQUET HALL
BAR OR TAVERN
BARBER SHOP/BEAUTY SALON
BARBERSHOP
BASEMENT
BEACH
BOAT / WATERCRAFT
BOAT/WATERCRAFT
BOWLING ALLEY
BRIDGE
CAR WASH
CASINO/GAMBLING ESTABLISHMENT
CEMETARY
CHA APARTMENT
CHA BREEZEWAY
CHA ELEVATOR
CHA GROUNDS
CHA HALLWAY
CHA HALLWAY / STAIRWELL / ELEVATOR
CHA HALLWAY/STAIRWELL/ELEVATOR
CHA LOBBY
CHA PARKING LOT
CHA PARKING LOT /

location_description
STREET                    2205639
RESIDENCE                 1367723
APARTMENT                 1004185
SIDEWALK                   760847
OTHER                      265268
                           ...   
COUNTY JAIL                     1
FUNERAL PARLOR                  1
ROOF                            1
POLICE FACILITY                 1
JUNK YARD/GARBAGE DUMP          1
Name: count, Length: 218, dtype: int64